In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
catalog_name = "ecommerce"

In [0]:
df = spark.table("ecommerce.bronze.brz_order_items")
df.show(10)

In [0]:
df.printSchema()

In [0]:
df.select("quantity").distinct().show()

In [0]:
df = df.withColumn("quantity", when(col("quantity") == "Two", 2).otherwise(col("quantity")).cast("int"))
df.show(10)

In [0]:
df = df.dropDuplicates(["customer_id", "item_seq"])

In [0]:
df.select("unit_price_currency").distinct().show()

In [0]:
df = df.withColumn("discount_pct", regexp_replace(col("discount_pct"), "%", "").cast("double"))
df.show(10)

In [0]:
df = df.withColumn("unit_price", regexp_replace(col("unit_price"), r"\$", "").cast("double"))
df.show(10)

In [0]:
df.printSchema()

In [0]:
df = df.withColumn("tax_amount", col("tax_amount").cast("double"))

In [0]:
df.show(10)

In [0]:
df.select("channel").distinct().show()

In [0]:
df = df.withColumn("channel", when(col("channel") == "web", "Website").when(col("channel") == "app", "Mobile").otherwise(col("channel")))
df.select("channel").distinct().show()

In [0]:
df = df.withColumn("processed_time", current_timestamp())

In [0]:
display(df.limit(5))

In [0]:
df.printSchema()

In [0]:
df.write.format("delta")\
        .mode("overwrite")\
        .option("mergeSchema", True)\
        .saveAsTable(f"{catalog_name}.silver.slv_order_items")